# AlphaCluster — Multi-Asset Training v4 (Active Trading)

Train a new model from scratch on **10 crypto assets** with **tuned reward function**
to encourage active trading instead of buy & hold.

**v4 changes vs v3:**
- Inactivity penalty (min 4 trades/episode target)
- Churn threshold reduced: 50 → 15 steps (penalize only scalping, not normal trades)
- Position hold bonus capped: 3.0 → 1.5 (less reward for holding forever)
- Trade completion duration divisor: 30 → 15, cap 10 → 4 (less bonus for long holds)
- Opportunity cost: scale 0.5 → 1.0, cap 0.02 → 0.03, threshold 0.002 → 0.0015

**Same as v3:** PPO, simple_actions=True (3-action: long/flat/short), fixed 10% size,
10x leverage, curriculum learning (Phase 1→2→3), VecNormalize(norm_reward=True),
4 parallel envs, all default PPO hyperparams.

**Budget:** 3M steps, ~5-6h training at ~150 fps. Eval + .pt checkpoint every 100k steps.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["PYTHONWARNINGS"] = "ignore"

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", message=".*Gym has been unmaintained.*")
warnings.filterwarnings("ignore", message=".*datetime.datetime.utcnow.*")

!pip install -q "stable-baselines3>=2.4,<3.0" "gymnasium>=1.0,<2.0" pyarrow python-dotenv tqdm rich

import sys
from pathlib import Path

# Add the project source to the Python path
SRC_DIR = "/kaggle/input/datasets/dawidgwczyk/alphacluster-repo/src"
sys.path.insert(0, SRC_DIR)

import alphacluster
print(f"AlphaCluster loaded from {Path(alphacluster.__file__).parent}")

import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected.")

## Load & Split Multi-Asset Data

Load all available `*_5m.parquet` files. Each asset is split chronologically
(70/15/15) and the training sets are passed as a list to `TradingEnv(dfs=...)`.

In [ ]:
import pandas as pd

DATA_DIR = Path("/kaggle/input/datasets/dawidgwczyk/alphacluster-data")

# Top 10 assets by market cap (balance between diversity and memory)
TOP_10 = [
    "btcusdt", "ethusdt", "bnbusdt", "solusdt", "xrpusdt",
    "dogeusdt", "adausdt", "avaxusdt", "trxusdt", "linkusdt",
]

all_dfs = {}
for symbol in TOP_10:
    path = DATA_DIR / f"{symbol}_5m.parquet"
    if not path.exists():
        print(f"  SKIP {symbol.upper()} (not found)")
        continue
    df = pd.read_parquet(path, engine="pyarrow")
    all_dfs[symbol.upper()] = df
    date_range = f"{df['open_time'].iloc[0]} to {df['open_time'].iloc[-1]}"
    print(f"  {symbol.upper()}: {len(df):>10,} candles  ({date_range})")

print(f"\nLoaded {len(all_dfs)} assets, {sum(len(d) for d in all_dfs.values()):,} total candles")

# Chronological split per asset: 70% train / 15% val / 15% test
train_dfs = []
val_dfs = []
test_dfs = []

for symbol, df in all_dfs.items():
    n = len(df)
    train_end = int(n * 0.70)
    val_end = int(n * 0.85)
    train_dfs.append(df.iloc[:train_end].reset_index(drop=True))
    val_dfs.append(df.iloc[train_end:val_end].reset_index(drop=True))
    test_dfs.append(df.iloc[val_end:].reset_index(drop=True))

total_train = sum(len(d) for d in train_dfs)
print(f"Train candles: {total_train:,}")
print(f"Episodes for 1x coverage: ~{total_train // 2016:,}")
print(f"Timesteps for 2x coverage: ~{total_train // 2016 * 2016 * 2:,}")

# BTC splits for evaluation (consistent with v3 baseline)
btc_df = all_dfs["BTCUSDT"]
n_btc = len(btc_df)
btc_val_df = btc_df.iloc[int(n_btc * 0.70):int(n_btc * 0.85)].reset_index(drop=True)
btc_test_df = btc_df.iloc[int(n_btc * 0.85):].reset_index(drop=True)
print(f"BTC eval: val={len(btc_val_df):,}  test={len(btc_test_df):,}")

# Load sentiment data for each asset
print("\nLoading sentiment data...")
sentiment_dfs_list = []
for symbol in [s.lower() for s in all_dfs.keys()]:
    sentiment = {}
    funding_path = DATA_DIR / f"{symbol}_funding.parquet"
    oi_path = DATA_DIR / f"{symbol}_oi.parquet"
    ls_path = DATA_DIR / f"{symbol}_ls_ratio.parquet"

    if funding_path.exists():
        sentiment["funding"] = pd.read_parquet(funding_path, engine="pyarrow")
        print(f"    + {symbol.upper()} funding: {len(sentiment['funding']):,} records")
    if oi_path.exists():
        sentiment["oi"] = pd.read_parquet(oi_path, engine="pyarrow")
        print(f"    + {symbol.upper()} OI: {len(sentiment['oi']):,} records")
    if ls_path.exists():
        sentiment["ls_ratio"] = pd.read_parquet(ls_path, engine="pyarrow")
        print(f"    + {symbol.upper()} L/S ratio: {len(sentiment['ls_ratio']):,} records")

    sentiment_dfs_list.append(sentiment)

train_sentiment = sentiment_dfs_list  # merge_asof handles alignment by timestamp
btc_sentiment = sentiment_dfs_list[0]  # BTC is first in TOP_10
print(f"\nSentiment data loaded for {len(sentiment_dfs_list)} assets")

## Train from Scratch (3M steps, 10 assets, v4 reward)

Uses `train()` from `trainer.py` — identical training pipeline to v3
(CurriculumCallback + TrainingMetricsCallback + EvalCallback).

Additional: EvalAndSaveCallback runs BTC backtest + saves .pt every 100k steps.

**Key reward changes:** inactivity penalty, reduced churn threshold (15), capped hold bonus (1.5),
reduced completion duration bonus, stronger opportunity cost.

In [ ]:
import csv
import time

import torch
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback, CallbackList
from stable_baselines3.common.vec_env import SubprocVecEnv, VecNormalize

from alphacluster.agent.config import TrainingConfig
from alphacluster.agent.trainer import create_agent, save_agent, train
from alphacluster.backtest.metrics import calculate_metrics
from alphacluster.backtest.runner import run_backtest
from alphacluster.config import MODEL_VERSION
from alphacluster.env.trading_env import TradingEnv

# ── Paths ──
MODELS_DIR = Path("/kaggle/working/models")
CHECKPOINT_DIR = MODELS_DIR / "checkpoints"
RESULTS_CSV = MODELS_DIR / "checkpoint_results.csv"
RESUME_MODEL = CHECKPOINT_DIR / "resume_model.zip"
RESUME_VECNORM = CHECKPOINT_DIR / "resume_vecnormalize.pkl"

FINETUNE_TIMESTEPS = 3_000_000
EVAL_FREQ = 100_000  # eval + save .pt every 100k
RESUME_SAVE_FREQ = 1_000_000  # save full SB3 state every 1M (for resume)
N_ENVS = 4
EVAL_EPISODES = 5
EVAL_SEED = 42


class ProgressCallback(BaseCallback):
    """Print a one-line progress update every log_interval timesteps."""

    def __init__(self, total_timesteps: int, log_interval: int = 50_000):
        super().__init__(verbose=0)
        self.total_timesteps = total_timesteps
        self.log_interval = log_interval
        self._start_time = None
        self._next_log = log_interval

    def _on_training_start(self) -> None:
        self._start_time = time.time()

    def _on_step(self) -> bool:
        if self.num_timesteps >= self._next_log:
            elapsed = time.time() - self._start_time
            pct = self.num_timesteps / self.total_timesteps * 100
            fps = self.num_timesteps / max(elapsed, 1)
            eta = (self.total_timesteps - self.num_timesteps) / max(fps, 1)
            print(
                f"  [{pct:5.1f}%] {self.num_timesteps:>10,} / {self.total_timesteps:,} "
                f"| {fps:.0f} fps | ETA {eta/60:.0f}m",
                flush=True,
            )
            self._next_log += self.log_interval
        return True


class EvalAndSaveCallback(BaseCallback):
    """Every eval_freq steps: run BTC backtest, save .pt checkpoint, append CSV.
    Every resume_save_freq steps: save full SB3 model + VecNormalize for resume."""

    def __init__(self, eval_freq, eval_env, checkpoint_dir, results_csv,
                 train_env_ref=None, resume_save_freq=1_000_000,
                 n_episodes=5, seed=42):
        super().__init__(verbose=0)
        self.eval_freq = eval_freq
        self.eval_env = eval_env
        self.checkpoint_dir = Path(checkpoint_dir)
        self.results_csv = Path(results_csv)
        self.train_env_ref = train_env_ref  # VecNormalize wrapper
        self.resume_save_freq = resume_save_freq
        self.n_episodes = n_episodes
        self.seed = seed
        self._next_eval = eval_freq
        self._next_resume_save = resume_save_freq
        self._results = []
        self._best_pnl = float("-inf")

    def _on_step(self) -> bool:
        # ── Full SB3 save for resume (every 1M) ──
        if self.num_timesteps >= self._next_resume_save:
            self._next_resume_save += self.resume_save_freq
            self.checkpoint_dir.mkdir(parents=True, exist_ok=True)
            resume_path = self.checkpoint_dir / "resume_model"
            self.model.save(resume_path)
            if self.train_env_ref is not None:
                self.train_env_ref.save(str(self.checkpoint_dir / "resume_vecnormalize.pkl"))
            print(f"  [RESUME CHECKPOINT saved at {self.num_timesteps//1000}k]", flush=True)

        # ── Eval + .pt save (every 100k) ──
        if self.num_timesteps < self._next_eval:
            return True
        self._next_eval += self.eval_freq

        step_k = self.num_timesteps // 1000

        # ── Evaluate ──
        result = run_backtest(
            model=self.model, env=self.eval_env,
            n_episodes=self.n_episodes, seed=self.seed,
        )
        m = calculate_metrics(result)

        row = {
            "timesteps": self.num_timesteps,
            "step": f"{step_k}k",
            "avg_pnl_pct": round(m["avg_episode_return_pct"], 2),
            "trades_per_ep": round(m["avg_trades_per_episode"], 1),
            "win_rate": round(m.get("win_rate", 0), 1),
            "profit_factor": round(m.get("profit_factor", 0), 4),
            "avg_trade_duration": round(m.get("avg_trade_duration", 0), 1),
            "max_drawdown_pct": round(m.get("max_drawdown_pct", 0), 2),
            "sharpe": round(m.get("sharpe_ratio", 0), 4),
            "avg_win": round(m.get("avg_win", 0), 2),
            "avg_loss": round(m.get("avg_loss", 0), 2),
            "long_pct": round(m.get("long_count", 0) / max(m.get("trade_count", 1), 1) * 100, 1),
            "flat_pct": round(m.get("avg_flat_pct", 0), 1),
            "fee_pnl_ratio": round(m.get("fee_to_pnl_ratio", 0), 1),
        }
        self._results.append(row)

        # ── Write CSV ──
        self.results_csv.parent.mkdir(parents=True, exist_ok=True)
        with open(self.results_csv, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=row.keys())
            writer.writeheader()
            writer.writerows(self._results)

        # ── Save .pt checkpoint ──
        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)
        pt_path = self.checkpoint_dir / f"checkpoint_{step_k}k.pt"
        torch.save(self.model.policy.state_dict(), pt_path)

        # ── Save best model ──
        is_best = ""
        if row["avg_pnl_pct"] > self._best_pnl:
            self._best_pnl = row["avg_pnl_pct"]
            best_path = self.checkpoint_dir / "best_model.pt"
            torch.save(self.model.policy.state_dict(), best_path)
            is_best = " ★ BEST"

        print(
            f"  [{row['step']}] PnL={row['avg_pnl_pct']:+.1f}% "
            f"trades={row['trades_per_ep']:.0f} WR={row['win_rate']:.0f}% "
            f"PF={row['profit_factor']:.2f} dur={row['avg_trade_duration']:.0f} "
            f"DD={row['max_drawdown_pct']:.0f}% long={row['long_pct']:.0f}%{is_best}",
            flush=True,
        )
        return True

    @property
    def results(self):
        return self._results


def make_env(dfs_list, sentiment_list, config, rank=0):
    """Factory for SubprocVecEnv with multi-asset support."""
    def _init():
        import os
        import warnings

        os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
        warnings.filterwarnings("ignore", category=DeprecationWarning)
        warnings.filterwarnings("ignore", message=".*Gym has been unmaintained.*")
        warnings.filterwarnings("ignore", message=".*datetime.datetime.utcnow.*")

        env = TradingEnv(
            dfs=dfs_list,
            sentiment_dfs=sentiment_list,
            window_size=config.window_size,
            episode_length=config.episode_length,
            simple_actions=config.simple_actions,
            fixed_size_pct=config.fixed_size_pct,
            fixed_leverage=config.fixed_leverage,
        )
        env.reset(seed=rank)
        return env
    return _init


# ── Config ──
config = TrainingConfig(
    total_timesteps=FINETUNE_TIMESTEPS,
    simple_actions=True,
    fixed_size_pct=0.10,
    fixed_leverage=10,
)
config.eval_freq = EVAL_FREQ

# ── Training environments (multi-asset) ──
print(f"Creating {N_ENVS} parallel training environments ({len(train_dfs)} assets) ...")
envs = SubprocVecEnv([make_env(train_dfs, train_sentiment, config, rank=i) for i in range(N_ENVS)])
train_env = VecNormalize(envs, norm_obs=False, norm_reward=True, clip_reward=10.0)

# Eval env: BTC validation set (for EvalCallback + TrainingMetricsCallback)
eval_env = TradingEnv(
    df=btc_val_df,
    funding_df=btc_sentiment.get("funding"),
    oi_df=btc_sentiment.get("oi"),
    ls_ratio_df=btc_sentiment.get("ls_ratio"),
    window_size=config.window_size,
    episode_length=config.episode_length,
    simple_actions=config.simple_actions,
    fixed_size_pct=config.fixed_size_pct,
    fixed_leverage=config.fixed_leverage,
)

# BTC test env (for checkpoint backtest — separate from training eval)
btc_test_env = TradingEnv(
    df=btc_test_df,
    funding_df=btc_sentiment.get("funding"),
    oi_df=btc_sentiment.get("oi"),
    ls_ratio_df=btc_sentiment.get("ls_ratio"),
    window_size=config.window_size,
    episode_length=config.episode_length,
    simple_actions=config.simple_actions,
    fixed_size_pct=config.fixed_size_pct,
    fixed_leverage=config.fixed_leverage,
)

# ── Create or resume agent ──
RESUMING = RESUME_MODEL.exists()
if RESUMING:
    print(f"RESUMING from {RESUME_MODEL}")
    agent = PPO.load(RESUME_MODEL, env=train_env)
    if RESUME_VECNORM.exists():
        train_env = VecNormalize.load(str(RESUME_VECNORM), envs)
        print(f"  VecNormalize stats restored from {RESUME_VECNORM}")
    remaining = FINETUNE_TIMESTEPS  # train() uses total_timesteps as remaining
    print(f"  Resuming training for {remaining:,} more steps")
else:
    print("Creating PPO agent (CNN+Transformer) from scratch ...")
    agent = create_agent(train_env, config, verbose=0)
    remaining = FINETUNE_TIMESTEPS

# ── Callbacks ──
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
progress_cb = ProgressCallback(remaining, log_interval=100_000)
eval_save_cb = EvalAndSaveCallback(
    eval_freq=EVAL_FREQ,
    eval_env=btc_test_env,
    checkpoint_dir=CHECKPOINT_DIR,
    results_csv=RESULTS_CSV,
    train_env_ref=train_env,
    resume_save_freq=RESUME_SAVE_FREQ,
    n_episodes=EVAL_EPISODES,
    seed=EVAL_SEED,
)

# ── Train ──
mode = "RESUMING" if RESUMING else "from scratch"
print(f"\nTraining {mode} for {remaining:,} steps (multi-asset)")
print(f"Model version: {MODEL_VERSION}")
print(f"Curriculum: Phase 1 (0-30%) -> Phase 2 (30-60%) -> Phase 3 (60-100%)")
print(f"Config: lr={config.learning_rate}, batch={config.batch_size}, gamma={config.gamma}")
print(f"VecNormalize: norm_obs=False, norm_reward=True, clip_reward=10.0")
print(f"Assets: {len(train_dfs)}, eval every {EVAL_FREQ//1000}k (.pt), resume save every {RESUME_SAVE_FREQ//1_000_000}M")
print(f"Reward v4: inactivity_penalty, churn(15), hold_cap=1.5, opp_cost=1.0, completion_cap=4")
print(f"Estimated time: ~{remaining / 150 / 3600:.0f}h at 150 fps\n")

start = time.time()
agent = train(
    agent=agent,
    config=config,
    eval_env=eval_env,
    checkpoint_dir=str(CHECKPOINT_DIR),
    run_tournament=False,
    progress_bar=False,
    verbose=0,
    extra_callbacks=[progress_cb, eval_save_cb],
)
elapsed = time.time() - start
print(f"\nTraining complete in {elapsed/60:.1f} minutes ({elapsed/3600:.1f}h)")

# ── Save final model (.pt for eval, .zip for resume) ──
MODELS_DIR.mkdir(parents=True, exist_ok=True)
final_pt = MODELS_DIR / "ppo_trading_multi_asset_final.pt"
torch.save(agent.policy.state_dict(), final_pt)
agent.save(RESUME_MODEL)
train_env.save(str(RESUME_VECNORM))
print(f"Final model: {final_pt} (eval) + {RESUME_MODEL} (resume)")

# ── Print best checkpoint ──
if eval_save_cb.results:
    import pandas as pd
    rdf = pd.DataFrame(eval_save_cb.results)
    best_idx = rdf["avg_pnl_pct"].idxmax()
    best = rdf.iloc[best_idx]
    print(f"Best checkpoint: {best['step']} — PnL={best['avg_pnl_pct']:+.1f}%")

train_env.close()

## Final Evaluation + Best Checkpoint Selection

Run full eval on BTC test set for the final model, compare with v3 baseline.

In [ ]:
from alphacluster.backtest.runner import run_backtest
from alphacluster.backtest.metrics import calculate_metrics, print_report

# ── Final model eval ──
test_env = TradingEnv(
    df=btc_test_df,
    funding_df=btc_sentiment.get("funding"),
    oi_df=btc_sentiment.get("oi"),
    ls_ratio_df=btc_sentiment.get("ls_ratio"),
    window_size=config.window_size,
    episode_length=config.episode_length,
    simple_actions=config.simple_actions,
    fixed_size_pct=config.fixed_size_pct,
    fixed_leverage=config.fixed_leverage,
)

print("Final model evaluation on BTC TEST set (10 episodes, seed=42):\n")
result = run_backtest(model=agent, env=test_env, n_episodes=10, seed=EVAL_SEED)
metrics = calculate_metrics(result)
print_report(metrics)

# ── Checkpoint progression ──
print("\n" + "=" * 80)
print("  CHECKPOINT PROGRESSION (BTC test set)")
print("=" * 80)

results_df = pd.read_csv(RESULTS_CSV)
# Show every 10th row for readability
display_df = results_df.iloc[::10].copy()
print(display_df[["step", "avg_pnl_pct", "trades_per_ep", "win_rate",
                   "profit_factor", "avg_trade_duration", "long_pct",
                   "max_drawdown_pct", "sharpe"]].to_string(index=False))

# ── Best checkpoint ──
best_idx = results_df["avg_pnl_pct"].idxmax()
best = results_df.iloc[best_idx]
print(f"\n  BEST CHECKPOINT: {best['step']} — "
      f"PnL={best['avg_pnl_pct']:+.1f}%, WR={best['win_rate']:.0f}%, "
      f"PF={best['profit_factor']:.2f}, trades/ep={best['trades_per_ep']:.0f}")

# ── Previous run comparison ──
print(f"\n  Previous best (best_model): PnL=-7.1%, trades/ep=49.8, WR=47.4%, PF=0.63")
print(f"  v3 baseline (500k, BTC-only): PnL=-0.95%, trades/ep=8.8, WR=40.9%, PF=0.886")
print(f"  Current best:                 PnL={best['avg_pnl_pct']:+.1f}%, "
      f"trades/ep={best['trades_per_ep']:.0f}, "
      f"WR={best['win_rate']:.0f}%, PF={best['profit_factor']:.2f}")
print("=" * 80)

## Training Progression Charts

In [ ]:
import matplotlib.pyplot as plt

if RESULTS_CSV.exists():
    df_r = pd.read_csv(RESULTS_CSV)
    steps = df_r["timesteps"] / 1_000_000

    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    fig.suptitle("Multi-Asset Training Progression (from scratch, 10 assets)", fontsize=14)

    plots = [
        ("avg_pnl_pct", "Avg PnL per Episode (%)"),
        ("trades_per_ep", "Trades per Episode"),
        ("win_rate", "Win Rate (%)"),
        ("profit_factor", "Profit Factor"),
        ("avg_trade_duration", "Avg Trade Duration (steps)"),
        ("sharpe", "Sharpe Ratio"),
    ]

    for ax, (col, title) in zip(axes.flat, plots):
        if col in df_r.columns:
            ax.plot(steps, df_r[col], marker="o", markersize=5, linewidth=2)
            # v3 baseline reference lines
            baselines = {
                "avg_pnl_pct": -0.95,
                "trades_per_ep": 8.8,
                "win_rate": 40.9,
                "profit_factor": 0.886,
                "avg_trade_duration": 228,
                "sharpe": -1.29,
            }
            if col in baselines:
                ax.axhline(y=baselines[col], color="red", linestyle="--",
                           alpha=0.6, label="v3 baseline")
                ax.legend(fontsize=8)
        ax.set_title(title)
        ax.set_xlabel("Steps (M)")
        ax.grid(True, alpha=0.3)

        # Mark curriculum phase boundaries
        for pct, label in [(0.3, "P2"), (0.6, "P3")]:
            ts = FINETUNE_TIMESTEPS * pct / 1_000_000
            ax.axvline(x=ts, color="blue", linestyle=":", alpha=0.4)
            ax.text(ts, ax.get_ylim()[1], f" {label}", color="blue", fontsize=7, va="top")

    plt.tight_layout()
    plt.show()
else:
    print("No checkpoint results found. Run training first.")

## Download

In [ ]:
import os

print("=" * 60)
print("  TRAINING COMPLETE")
print("=" * 60)
print()
print("Model files in /kaggle/working/ (download from Output tab):")
print()

for root, dirs, files in os.walk("/kaggle/working/models"):
    level = root.replace("/kaggle/working/", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    sub_indent = "  " * (level + 1)
    for f in sorted(files):
        size_mb = os.path.getsize(os.path.join(root, f)) / (1024 * 1024)
        print(f"{sub_indent}{f}  ({size_mb:.1f} MB)")

print()
print("To evaluate locally:")
print("  python scripts/evaluate.py --model-path models/ppo_trading_multi_asset_final.pt --simple-actions")